# 滑动窗口最大值：从建模到单调队列

这份 notebook 不直接给完整题解，而是把思考过程拆开：先把题目翻译成数学对象，再观察哪些信息可以复用，最后自然推出“单调队列”这个数据结构。

你可以按顺序运行代码单元，遇到 `TODO` 先自己填，再看后面的提示。

## 0. 题目建模

给定数组 `nums` 和窗口大小 `k`。第 `i` 个窗口可以写成一个闭区间：

$$
W_i = [i, i+k-1]
$$

对应答案是一个函数：

$$
ans_i = \max\{nums[j] \mid j \in W_i\}
$$

所以题目不是“不断找最大值”这么简单，而是一个特殊的区间查询问题：所有区间长度相同，并且右端点每次只移动 1。

In [1]:
nums = [1, 3, -1, -3, 5, 3, 6, 7]
k = 3

def windows(nums, k):
    return [(i, nums[i:i+k]) for i in range(len(nums) - k + 1)]

for i, w in windows(nums, k):
    print(f"W_{i} = nums[{i}:{i+k}] = {w}, max = {max(w)}")

W_0 = nums[0:3] = [1, 3, -1], max = 3
W_1 = nums[1:4] = [3, -1, -3], max = 3
W_2 = nums[2:5] = [-1, -3, 5], max = 5
W_3 = nums[3:6] = [-3, 5, 3], max = 5
W_4 = nums[4:7] = [5, 3, 6], max = 6
W_5 = nums[5:8] = [3, 6, 7], max = 7


## 1. 先接受暴力解，再问它浪费在哪里

最直接的做法：每个窗口都扫一遍，时间复杂度是：

$$
O((n-k+1) \cdot k) \approx O(nk)
$$

这一步很重要。**很多高效算法不是凭空来的，而是先写出“正确但慢”的模型，再找重复计算**。

In [2]:
def brute_force_window_max(nums, k):
    """正确但慢：作为后续练习的对照答案。"""
    if k <= 0 or k > len(nums):
        return []
    return [max(nums[i:i+k]) for i in range(len(nums) - k + 1)]

expected = brute_force_window_max(nums, k)
expected

[3, 3, 5, 5, 6, 7]

## 2. 相邻窗口只差两个元素

从 `W_i` 到 `W_{i+1}`：

- 离开一个元素：`nums[i]`
- 进入一个元素：`nums[i+k]`
- 中间的 `k-1` 个元素保持不变

如果上一个窗口最大值没有离开，似乎可以复用；如果它离开了，就必须知道剩余元素里的最大值。关键问题变成：我们该维护哪些“有希望成为最大值”的候选者？

In [3]:
def compare_neighbors(nums, k):
    for i in range(len(nums) - k):
        left = nums[i:i+k]
        right = nums[i+1:i+k+1]
        print(f"W_{i}   {left}  max={max(left)}")
        print(f"W_{i+1} {right}  max={max(right)}")
        print(f"leave={nums[i]}, enter={nums[i+k]}\n")

compare_neighbors(nums, k)

W_0   [1, 3, -1]  max=3
W_1 [3, -1, -3]  max=3
leave=1, enter=-3

W_1   [3, -1, -3]  max=3
W_2 [-1, -3, 5]  max=5
leave=3, enter=5

W_2   [-1, -3, 5]  max=5
W_3 [-3, 5, 3]  max=5
leave=-1, enter=3

W_3   [-3, 5, 3]  max=5
W_4 [5, 3, 6]  max=6
leave=-3, enter=6

W_4   [5, 3, 6]  max=6
W_5 [3, 6, 7]  max=7
leave=5, enter=7



## 3. 支配关系：为什么有些元素可以丢掉

考虑两个下标 `i < j`。如果：

$$
nums[i] \le nums[j]
$$

那么在所有同时包含 `i` 和 `j` 的未来窗口里，`i` 都不可能比 `j` 更适合作为最大值候选，因为：

- `j` 的值不小于 `i`
- `j` 更靠右，会比 `i` 更晚离开窗口

这就是本题的核心数学观察：**旧且不更大的元素，被新且更大的元素支配，可以永久删除。**

In [5]:
def dominated_pairs(nums):
    pairs = []
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] <= nums[j]:
                pairs.append((i, j, nums[i], nums[j]))
    return pairs

dominated_pairs([2, 1, 3, 2])[:8]

[(0, 2, 2, 3), (0, 3, 2, 2), (1, 2, 1, 3), (1, 3, 1, 2)]

思考题：如果 `nums[i] == nums[j]`，为什么通常也可以删除更左边的 `i`？

提示：它们值相同，但右边的 `j` 生命周期更长。

## 4. 候选集合应该满足的不变量

我们想维护一个候选下标序列 `q`，它需要同时满足两个顺序：

1. **下标从左到右递增**：这样能判断谁已经滑出窗口。
2. **对应的值从大到小单调不增**：这样队首永远是当前窗口最大值。

这就是“单调队列”的来源。它不是为了背模板而存在，而是由“窗口过期”和“最大值候选”两个需求共同推出来的。

In [6]:
from collections import deque

def describe_queue(nums, q):
    return [(idx, nums[idx]) for idx in q]

# 例子：如果 q = [1, 4, 5]，它表示这些下标仍然可能成为当前或未来窗口最大值。
# 是否合法，要同时看下标顺序和值顺序。
q = deque([1, 4, 5])
nums = [1, 3, -1, -3, 5, 3, 6, 7]
describe_queue(nums, q)

[(1, 3), (4, 5), (5, 3)]

## 5. 先写“单步更新”，不要急着写完整函数

处理新下标 `right` 时，单调队列只做三类事：

1. 删除已经不在窗口 `[right-k+1, right]` 里的队首。
2. 从队尾删除被 `nums[right]` 支配的候选者。
3. 把 `right` 加到队尾。

下面的函数故意留空。先把这一小步写对，再组合成完整算法。

In [ ]:
def push_one(nums, q, right, k):
    """把下标 right 加入单调队列 q。

    练习目标：维护两个不变量：
    - q 中下标都在当前窗口内
    - nums[q[0]], nums[q[1]], ... 单调不增
    """
    left = right - k + 1

    # TODO 1: 如果 q[0] < left，说明队首已经过期，应该从左侧弹出。
    # while q and ...:
    #     ...

    # TODO 2: 如果队尾的值 <= nums[right]，队尾会被 right 支配，应该弹出。
    # while q and ...:
    #     ...

    # TODO 3: 把 right 加到队尾。
    # ...

    return q

### 自测 1：队列不变量

填好上面的 `push_one` 后，再运行下面的检查。

In [7]:
def check_invariants(nums, q, right, k):
    left = right - k + 1
    assert all(left <= idx <= right for idx in q), "有下标不在当前窗口内"
    assert all(q[i] < q[i+1] for i in range(len(q)-1)), "下标没有递增"
    assert all(nums[q[i]] >= nums[q[i+1]] for i in range(len(q)-1)), "值没有单调不增"

q = deque()
for right in range(len(nums)):
    push_one(nums, q, right, k)
    check_invariants(nums, q, right, k)
    print(f"right={right}, window={nums[max(0, right-k+1):right+1]}, q={describe_queue(nums, q)}")

NameError: name 'push_one' is not defined

## 6. 从单步更新组合出完整过程

当 `right >= k-1` 时，窗口长度已经达到 `k`，这时 `q[0]` 就是当前窗口最大值的下标。

这一步不需要新的思想，只是把“单步更新”和“记录答案”拼起来。

In [ ]:
def guided_window_max(nums, k):
    """练习版：依赖你在 push_one 中维护好单调队列。"""
    if k <= 0 or k > len(nums):
        return []

    q = deque()
    ans = []
    for right in range(len(nums)):
        push_one(nums, q, right, k)

        # TODO 4: 当形成完整窗口后，把当前最大值加入 ans。
        # 条件：right >= k - 1
        # 当前最大值：nums[q[0]]
        # if ...:
        #     ...

    return ans

In [ ]:
# 填好 TODO 后，下面应该通过。
assert guided_window_max(nums, k) == expected
assert guided_window_max([1], 1) == [1]
assert guided_window_max([9, 8, 7], 2) == [9, 8]
assert guided_window_max([1, 2, 3], 2) == [2, 3]
assert guided_window_max([2, 2, 2], 2) == [2, 2]
"all checks passed"

## 7. 为什么总复杂度是 O(n)

单调队列里看起来有 `while`，但它不是 `O(nk)`。

摊还分析的关键：

- 每个下标最多从右侧进入队列一次。
- 每个下标最多从左侧因为过期弹出一次。
- 每个下标最多从右侧因为被支配弹出一次。

所以所有弹出次数加起来不超过 `n` 级别，总时间复杂度是 `O(n)`，空间复杂度是 `O(k)`。

In [ ]:
def count_bruteforce_work(n, k):
    return max(0, n - k + 1) * k

def count_monotonic_queue_bound(n):
    # 粗略上界：每个元素入队一次、出队最多一次；常数不重要，数量级重要。
    return 2 * n

for n, k in [(8, 3), (100, 10), (100_000, 500)]:
    print(f"n={n:>6}, k={k:>3} | brute ~ {count_bruteforce_work(n, k):>10} ops | mono queue ~ {count_monotonic_queue_bound(n):>10} ops")

## 8. 举一反三：什么时候想到单调队列

可以用下面的问题检查一个题是否适合单调队列：

1. 是否在维护一个连续区间或时间窗口？
2. 是否每一步只新增一个元素、删除一批过期元素？
3. 目标是否是窗口中的最大值、最小值，或能用“支配关系”删掉一批候选？
4. 候选者是否既需要按时间顺序过期，又需要按值顺序快速取最优？

如果答案大多是“是”，就应该尝试建立一个按下标有序、按值单调的候选队列。

类似题型：

- 滑动窗口最小值：把队列维护成单调不减。
- 长度限制下的前缀和最优值：维护前缀和候选，常见于“最短子数组和至少为 K”。
- 动态规划优化：当状态转移只看最近 `k` 个状态里的最大或最小时。

## 9. 复盘问题

写完后，建议你用自己的话回答这几个问题：

1. 为什么队首一定是当前窗口最大值？
2. 为什么从队尾删除元素不会丢失正确答案？
3. 为什么相等时可以删除更早的元素？
4. `while` 循环为什么不会导致 `O(nk)`？
5. 如果题目要求窗口最小值，哪些比较符号要反过来？